# Génération de syllabes

Écoute et comparaison de trois sources de syllabes de canari, par type (les 16 types de Pagliarini et al. 2021 : A, B1, B2, C, D, E, H, J1, J2, L, M, N, O, Q, R, V) :

1. **Vraies syllabes** : celles de marron1 sur lesquelles le GAN est entraîné (`canopy.syllable_generation.syllables` : découpées dans les phrases des chants d'entraînement, 1000 par type, filtrées au-dessus de 700 Hz, suivies de silence jusqu'à ~1 s à 16 kHz).
2. **Syllabes du GAN** : le WaveGAN à espace latent de dimension 3 de Pagliarini, Trouvain, Leblois & Hinaut (2021), « What does the canary say? Low-dimensional GAN applied to birdsong » (`canopy.syllable_generation.wavegan`), à partir de vecteurs latents tirés au hasard dans [-1, 1]³ ; leur type est celui que leur donne le **décodeur** (`canopy.syllable_generation.decoder` : un ESN sur les MFCC, le modèle perceptif de l'article suivant, avec le type poubelle X).
3. **Productions du modèle sensori-moteur** de Pagliarini, Leblois & Hinaut (2021), « Canary vocal sensorimotor model with RNN decoder and low-dimensional GAN generator » (`canopy.syllable_generation.sensorimotor`) : un modèle inverse, du type de syllabe voulu vers une commande motrice (un point de l'espace latent du GAN), appris par une règle hebbienne à partir d'un babillage aléatoire écouté par le décodeur.

Les GAN sont entraînés sur PlaFRIM (`slurm/submit_gan.sh`, un checkpoint toutes les 15 époques). Rapatrier les pertes et les checkpoints utiles (environ 70 Mo chacun ; les époques 15, 30 et 45 servent aussi d'exemples du type poubelle du décodeur), par exemple :

```bash
rsync -av --include='losses.csv' --include='checkpoints/' --include='epoch_00[0-4]?.pt' --include='epoch_0990.pt' --exclude='*' plafrim:/beegfs/tcazalet/canopy_gan/outputs/canary/syllable_gan/wavegan_0/ outputs/canary/syllable_gan/wavegan_0/
```

## Réglages

Le run du GAN (un dossier de `outputs/canary/syllable_gan`) et l'époque de son générateur écoutés par les sections 2 et 3, choisis dans les listes ; exécuter ensuite les cellules suivantes.

In [ ]:
import numpy as np  # (before torch: the reverse order loads two OpenMP runtimes in hag_env)
import ipywidgets as widgets
from IPython.display import display

from canopy.common import OUTPUTS

RUNS = OUTPUTS / "syllable_gan"
runs = sorted(p.parent.name for p in RUNS.glob("*/checkpoints") if any(p.glob("epoch_*.pt")))


def epochs_of(run):
    return [int(p.stem.split("_")[1]) for p in sorted((RUNS / run / "checkpoints").glob("epoch_*.pt"))]


run_choice = widgets.Dropdown(options=runs, description="run")
epoch_choice = widgets.Dropdown(description="époque")


def update_epochs(change=None):
    epoch_choice.options = epochs_of(run_choice.value) if run_choice.value else []
    epoch_choice.value = epoch_choice.options[-1] if epoch_choice.options else None


run_choice.observe(update_epochs, "value")
update_epochs()
display(widgets.HBox([run_choice, epoch_choice]) if runs else
        widgets.HTML(f"Aucun checkpoint dans {RUNS} : rapatrier un run de PlaFRIM (commande ci-dessus)."))

## Écoute

Les affichages des cellules suivantes : pour un type choisi dans la liste, `EXAMPLES` exemples tirés au hasard, chacun avec son spectrogramme (les `SHOWN` premières secondes) et un lecteur (le son normalisé). `listen` montre une liste de sons avec leurs titres.

In [ ]:
import io

import librosa
import matplotlib.pyplot as plt
from IPython.display import Audio

from canopy.syllable_generation.syllables import LABELS, SAMPLING_RATE

EXAMPLES = 6
SHOWN = 0.4  # s


def spectrogram_png(x):
    S = librosa.amplitude_to_db(np.abs(librosa.stft(x[:int(SHOWN * SAMPLING_RATE)], n_fft=256, hop_length=32)),
                                ref=np.max)
    fig, ax = plt.subplots(figsize=(2.2, 1.6))
    ax.imshow(S, origin="lower", aspect="auto", vmin=-60, cmap="magma", extent=[0, SHOWN * 1000, 0, 8])
    ax.set_xlabel("ms", fontsize=7)
    ax.set_ylabel("kHz", fontsize=7)
    ax.tick_params(labelsize=7)
    fig.tight_layout(pad=0.2)
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=80)
    plt.close(fig)
    return buffer.getvalue()


def sound_box(x, title):
    player = widgets.Output()
    with player:
        display(Audio(x[:int(SHOWN * SAMPLING_RATE)], rate=SAMPLING_RATE, normalize=True))
    return widgets.VBox([widgets.HTML(f"<b>{title}</b>"), widgets.Image(value=spectrogram_png(x)), player],
                        layout=widgets.Layout(width="230px"))


def listen(audio, titles, per_row=EXAMPLES):
    boxes = [sound_box(x, t) for x, t in zip(audio, titles)]
    display(widgets.VBox([widgets.HBox(boxes[k:k + per_row]) for k in range(0, len(boxes), per_row)]))


def by_type(audio, types, scores=None, seed=0, labels=LABELS):
    """Player of EXAMPLES sounds of audio of the type chosen in the list (types: the type of each sound; scores: shown
    with the titles)."""
    choice, shown = widgets.Dropdown(options=labels, description="type"), widgets.Output()
    rng = np.random.default_rng(seed)

    def show(change=None):
        with shown:
            shown.clear_output(wait=True)
            index = np.flatnonzero(types == choice.value)
            if len(index) == 0:
                print(f"aucun son de type {choice.value}")
                return
            index = rng.choice(index, min(EXAMPLES, len(index)), replace=False)
            listen(audio[index], [f"{choice.value} #{i}" + (f" ({scores[i]:.2f})" if scores is not None else "")
                                  for i in index])

    choice.observe(show, "value")
    show()
    display(widgets.VBox([choice, shown]))

## 1. Vraies syllabes

Les syllabes d'entraînement du GAN (1000 par type, chants d'entraînement de marron1).

In [ ]:
from canopy.syllable_generation.syllables import syllable_dataset

real = syllable_dataset("train")
real_types = real["labels"][real["label"]]
by_type(real["x"], real_types)

## 2. Syllabes du GAN

### Entraînement

Pertes du run choisi (`losses.csv`) : distance de Wasserstein estimée par le critique (elle doit décroître quand les sons générés se rapprochent des vrais), pénalité de gradient et perte du générateur.

In [ ]:
import pandas as pd

run_dir = RUNS / run_choice.value
losses = pd.read_csv(run_dir / "losses.csv").drop_duplicates("step", keep="last")
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, column in zip(axes, ["wasserstein", "gradient_penalty", "generator_loss"]):
    ax.plot(losses["epoch"], losses[column], lw=0.8)
    ax.plot(losses["epoch"], losses[column].rolling(25, min_periods=1).mean(), lw=1.5)
    ax.set_title(column)
    ax.set_xlabel("époque")
plt.tight_layout()
plt.show()

### Décodeur

Le décodeur du type des sons (`Decoder`, ESN de `UNITS` neurones sur les MFCC, entraîné à donner le type à chaque trame) : entraîné sur les vraies syllabes d'entraînement (10 % gardées pour choisir le coefficient de ridge) et, comme le décodeur « EXT » de Pagliarini et al., sur un type poubelle X : du bruit blanc et des sons du GAN aux époques `GARBAGE_EPOCHS` (pas encore réalistes), `GARBAGE_PER_SOURCE` de chaque. Mis en cache dans le dossier du run (`decoder.pkl`). Sa précision sur les syllabes des chants de test (jamais vues par le GAN ni par le décodeur) : par type et globale.

In [ ]:
import pickle

from canopy.syllable_generation.decoder import GARBAGE, Decoder
from canopy.syllable_generation.wavegan import LATENT_DIM, generate, load_generator

GARBAGE_EPOCHS = (15, 30, 45)
GARBAGE_PER_SOURCE = 250


def garbage_sounds(run, rng):
    sounds = [rng.uniform(-1, 1, (GARBAGE_PER_SOURCE, 1)) * rng.normal(0, 1, (GARBAGE_PER_SOURCE, real["x"].shape[1]))
              * 0.3]
    for epoch in GARBAGE_EPOCHS:
        checkpoint = RUNS / run / "checkpoints" / f"epoch_{epoch:04d}.pt"
        if checkpoint.exists():
            z = rng.uniform(-1, 1, (GARBAGE_PER_SOURCE, LATENT_DIM)).astype(np.float32)
            sounds.append(generate(load_generator(checkpoint), z))
        else:
            print(f"pas de checkpoint à l'époque {epoch} : pas de sons du GAN de cette époque dans le type poubelle")
    return np.clip(np.concatenate(sounds), -1, 1).astype(np.float32)


decoder_file = run_dir / "decoder.pkl"
if decoder_file.exists():
    with open(decoder_file, "rb") as f:
        decoder = pickle.load(f)
else:
    rng = np.random.default_rng(0)
    garbage = garbage_sounds(run_choice.value, rng)
    x = np.concatenate([real["x"], garbage])
    labels = np.concatenate([real_types, np.full(len(garbage), GARBAGE)])
    validation = np.zeros(len(x), bool)
    validation[rng.choice(len(x), len(x) // 10, replace=False)] = True
    decoder = Decoder(seed=0).fit(x[~validation], labels[~validation], x[validation], labels[validation])
    with open(decoder_file, "wb") as f:
        pickle.dump(decoder, f)

test = syllable_dataset("test", per_label=None)
test_types = test["labels"][test["label"]]
predicted = decoder.predict(test["x"])
per_type = pd.Series({c: np.mean(predicted[test_types == c] == c) for c in LABELS})
print(f"décodeur : ridge {decoder.ridge:g}, précision sur les syllabes des chants de test {np.mean(predicted == test_types):.1%} "
      f"(moyenne par type {per_type.mean():.1%}, {np.mean(predicted == GARBAGE):.1%} classées X)")
per_type.plot.bar(figsize=(8, 2.5), ylim=(0, 1), title="précision du décodeur par type (vraies syllabes de test)")
plt.show()

### Sons générés

`N_GENERATED` sons du générateur de l'époque choisie, à partir de vecteurs latents tirés uniformément dans [-1, 1]³, et le type que leur donne le décodeur (sa sortie moyenne la plus forte ; entre parenthèses dans les titres : cette sortie). Répartition des types (un GAN qui couvre tout le répertoire en produit de chaque type), position des types dans l'espace latent (chaque point un vecteur latent, coloré par le type de son son), et écoute par type.

In [ ]:
import plotly.express as px

from canopy.common import label_colors

N_GENERATED = 2000

generator = load_generator(run_dir / "checkpoints" / f"epoch_{epoch_choice.value:04d}.pt")
z = np.random.default_rng(1).uniform(-1, 1, (N_GENERATED, LATENT_DIM)).astype(np.float32)
generated = generate(generator, z)
mean_outputs = decoder.raw(generated).mean(axis=1)
generated_types = decoder.classes[mean_outputs.argmax(axis=1)]
generated_scores = mean_outputs.max(axis=1)

counts = pd.Series(generated_types).value_counts().reindex(list(LABELS) + [GARBAGE], fill_value=0)
counts.plot.bar(figsize=(8, 2.5), title=f"types des {N_GENERATED} sons générés (époque {epoch_choice.value})")
plt.axhline(N_GENERATED / (len(LABELS) + 1), color="gray", lw=0.8, ls="--")
plt.show()

colors = label_colors(list(LABELS) + [GARBAGE])
px.scatter_3d(pd.DataFrame(dict(z1=z[:, 0], z2=z[:, 1], z3=z[:, 2], type=generated_types)), x="z1", y="z2", z="z3",
              color="type", color_discrete_map=colors, category_orders=dict(type=list(LABELS) + [GARBAGE]),
              opacity=0.6, height=550, title="espace latent : type du son de chaque vecteur").update_traces(
    marker_size=3).show()

In [ ]:
by_type(generated, generated_types, generated_scores, labels=list(LABELS) + [GARBAGE])

## 3. Modèle sensori-moteur (Pagliarini, Leblois & Hinaut 2021)

**Babillage** : `N_EXPLORATION` commandes motrices z tirées dans [-1, 1]³ (16 000 dans l'article), leurs sons G(z) écoutés par le décodeur. **Activation auditive** d'un son : les sorties du décodeur à la trame de sa sortie la plus forte dans la première demi-seconde, ramenées dans [0, 1], celle de chaque type divisée par son 95ᵉ centile sur le babillage (et plafonnée à 1). **Règle hebbienne** : W (3 × 16), initialisé près de 0, reçoit η z Aᵀ à chaque essai (un son du babillage tiré au hasard). **Production** du type s : le son du GAN à la commande clip(W[:, s], -1, 1), noté par son activation auditive du type s (1 : aussi reconnu que les 5 % des sons du babillage les mieux reconnus de ce type). Réglages de l'article : `TRIALS` = 3001 essais, η = 0,1 et 0,01.

In [ ]:
from canopy.syllable_generation import sensorimotor

N_EXPLORATION = 4000
ETAS = (0.1, 0.01)
EVALUATE_EVERY = 100  # trials

z_exploration, A, scaling = sensorimotor.explore(generator, decoder, N_EXPLORATION)
learning = {}
for eta in ETAS:
    W = sensorimotor.learn(z_exploration, A, eta)
    trials = np.arange(0, len(W), EVALUATE_EVERY)
    learning[eta] = dict(W=W, trials=trials, scores=np.array([sensorimotor.produce(W[t], generator, decoder, scaling)[1]
                                                              for t in trials]))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.2))
for eta, result in learning.items():
    axes[0].plot(result["trials"], result["scores"].mean(axis=1), label=f"η = {eta}")
axes[0].set(xlabel="essai", ylabel="score moyen des 16 types", ylim=(0, 1.05), title="apprentissage")
axes[0].legend()
final = pd.DataFrame({f"η = {eta}": result["scores"][-1] for eta, result in learning.items()}, index=LABELS)
final.plot.bar(ax=axes[1], ylim=(0, 1.05), title="score de chaque type à la fin")
plt.tight_layout()
plt.show()

### Productions apprises

Les 16 productions à la fin de l'apprentissage (η choisi dans `ETA`), avec leur score, puis les trajectoires des commandes motrices clip(W[:, s], -1, 1) au cours des essais dans l'espace latent (une couleur par type ; en gris : les sons du babillage reconnus comme ce type, à activation au moins 0,9).

In [ ]:
ETA = 0.01

W_final = learning[ETA]["W"][-1]
produced, scores = sensorimotor.produce(W_final, generator, decoder, scaling)
listen(produced, [f"{s} ({score:.2f})" for s, score in zip(LABELS, scores)])

In [ ]:
import plotly.graph_objects as go

W_path = np.clip(learning[ETA]["W"][::10], -1, 1)
figure = go.Figure()
for s, label in enumerate(LABELS):
    recognized = z_exploration[A[:, s] >= 0.9]
    figure.add_trace(go.Scatter3d(x=recognized[:, 0], y=recognized[:, 1], z=recognized[:, 2], mode="markers",
                                  marker=dict(size=2, color=colors[label], opacity=0.15), legendgroup=label,
                                  showlegend=False, hoverinfo="skip"))
    figure.add_trace(go.Scatter3d(x=W_path[:, 0, s], y=W_path[:, 1, s], z=W_path[:, 2, s], mode="lines+markers",
                                  marker=dict(size=2), line=dict(color=colors[label], width=4), name=label,
                                  legendgroup=label))
figure.update_layout(height=600, title=f"trajectoires des commandes motrices (η = {ETA})",
                     scene=dict(xaxis_range=[-1, 1], yaxis_range=[-1, 1], zaxis_range=[-1, 1]))
figure.show()